python 3.11, miniconda, cuda 12.8
ffmpeg==8.0.1
torch==2.9.0
torchcodec==0.9.1

```bash
conda install conda-forge::ffmpeg==8.0.1
ffmpeg -decoders | grep -i nvidia
```
Must be:
```bash
.....
  V..... h264_cuvid
.....
```

```bash
pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu128
pip install torchcodec==0.9.1 --index-url=https://download.pytorch.org/whl/cu128
```


In [1]:
from typing import Optional

import torch
from torch import nn
from torchaudio import functional as F

### Base

In [2]:
class LogMelFilterBanks(nn.Module):
    def __init__(
            self,
            n_fft: int = 400,
            samplerate: int = 16000,
            hop_length: int = 160,
            n_mels: int = 80,
            pad_mode: str = 'reflect',
            power: float = 2.0,
            normalize_stft: bool = False,
            onesided: bool = True,
            center: bool = True,
            return_complex: bool = True,
            f_min_hz: float = 0.0,
            f_max_hz: Optional[float] = None,
            norm_mel: Optional[str] = None,
            mel_scale: str = 'htk'
        ):
        super(LogMelFilterBanks, self).__init__()
        # general params and params defined by the exercise
        self.n_fft = n_fft
        self.samplerate = samplerate
        self.window_length = n_fft
        self.window = torch.hann_window(self.window_length)
        # Do correct initialization of stft params below:
        # hop_length, n_mels, center, return_complex, onesided, normalize_stft, pad_mode, power
        # ...
        # <YOUR CODE GOES HERE>

        # Do correct initialization of mel fbanks params below:
        # f_min_hz, f_max_hz, norm_mel, mel_scale
        # ...
        # <YOUR CODE GOES HERE>

        # finish parameters initialization
        self.mel_fbanks = self._init_melscale_fbanks()

    def _init_melscale_fbanks(self):
        # To access attributes, use self.<parameter_name>
        return F.melscale_fbanks(
            # Turns a normal STFT into a mel frequency STFT with triangular filter banks
            # make a full and correct function call
            # <YOUR CODE GOES HERE>
        )

    def spectrogram(self, x):
        # x - is an input signal
        return torch.stft(
            # make a full and correct function call
            # <YOUR CODE GOES HERE>
        )

    def forward(self, x):
        """
        Args:
            x (Torch.Tensor): Tensor of audio of dimension (batch, time), audiosignal
        Returns:
            Torch.Tensor: Tensor of log mel filterbanks of dimension (batch, n_mels, n_frames),
                where n_frames is a function of the window_length, hop_length and length of audio
        """
        # <YOUR CODE GOES HERE>
        # Return log mel filterbanks matrix
        return

### Impl

In [6]:
class LogMelFilterBanks(nn.Module):
    def __init__(
            self,
            n_fft: int = 400,
            samplerate: int = 16000,
            hop_length: int = 160,
            n_mels: int = 80,
            pad_mode: str = 'reflect',
            power: float = 2.0,
            normalize_stft: bool = False,
            onesided: bool = True,
            center: bool = True,
            return_complex: bool = True,
            f_min_hz: float = 0.0,
            f_max_hz: Optional[float] = None,
            norm_mel: Optional[str] = None,
            mel_scale: str = 'htk'
        ):
        super(LogMelFilterBanks, self).__init__()
        # general params and params defined by the exercise
        self.n_fft = n_fft
        self.samplerate = samplerate
        self.window_length = n_fft
        self.window = torch.hann_window(self.window_length)

        # Do correct initialization of stft params below:
        # hop_length, n_mels, center, return_complex, onesided, normalize_stft, pad_mode, power
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.center = center
        self.return_complex = return_complex
        self.onesided = onesided
        self.normalize_stft = normalize_stft
        self.pad_mode = pad_mode
        self.power = power

        # Do correct initialization of mel fbanks params below:
        # f_min_hz, f_max_hz, norm_mel, mel_scale
        self.f_min_hz = f_min_hz
        self.f_max_hz = f_max_hz if f_max_hz is not None else samplerate / 2
        self.norm_mel = norm_mel
        self.mel_scale = mel_scale

        # finish parameters initialization
        self.mel_fbanks = self._init_melscale_fbanks()

    def _init_melscale_fbanks(self):
        # To access attributes, use self.<parameter_name>
        n_freqs = self.n_fft // 2 + 1 if self.onesided else self.n_fft
        return F.melscale_fbanks(
            n_freqs=n_freqs,
            f_min=self.f_min_hz,
            f_max=self.f_max_hz,
            n_mels=self.n_mels,
            sample_rate=self.samplerate,
            norm=self.norm_mel,
            mel_scale=self.mel_scale
        )

    def spectrogram(self, x):
        # x - is an input signal
        window = self.window.to(x.device)
        stft = torch.stft(
            x,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.window_length,
            window=window,
            center=self.center,
            pad_mode=self.pad_mode,
            normalized=self.normalize_stft,
            onesided=self.onesided,
            return_complex=self.return_complex
        )
        if self.return_complex:
            power_spec = stft.abs() ** self.power
        else:
            power_spec = (stft[..., 0] ** 2 + stft[..., 1] ** 2) ** (self.power / 2)
        return power_spec

    def forward(self, x):
        """
        Args:
            x (Torch.Tensor): Tensor of audio of dimension (batch, time), audiosignal
        Returns:
            Torch.Tensor: Tensor of log mel filterbanks of dimension (batch, n_mels, n_frames),
                where n_frames is a function of the window_length, hop_length and length of audio
        """
        spec = self.spectrogram(x)                     # (batch, freq, time)
        mel_fbanks = self.mel_fbanks.to(x.device)      # (freq, n_mels)
        mel_spec = torch.einsum("bft,fm->bmt", spec, mel_fbanks)  # (batch, n_mels, time)
        eps = 1e-6
        log_mel = torch.log(mel_spec + eps)
        return log_mel

In [9]:
import torchaudio
signal, sr = torchaudio.load("pikuniku_pikuniku_dance.wav")

In [10]:
melspec = torchaudio.transforms.MelSpectrogram(
    hop_length=160,
    n_mels=80
)(signal)
logmelbanks = LogMelFilterBanks()(signal)

In [11]:
logmelbanks

tensor([[[-13.8155, -13.8155, -13.8155,  ..., -11.2408, -13.7529, -13.5992],
         [-13.8155, -13.8155, -13.8155,  ..., -10.0110, -13.6052, -13.1874],
         [-13.8155, -13.8155, -13.8155,  ..., -10.0749, -13.6476, -13.7040],
         ...,
         [-13.8155, -13.8155, -13.8155,  ..., -13.7447, -13.6932, -13.7653],
         [-13.8155, -13.8155, -13.8155,  ..., -13.7767, -13.7830, -13.7996],
         [-13.8155, -13.8155, -13.8155,  ..., -13.7032, -13.7614, -13.7585]],

        [[-13.8155, -13.8155, -13.8155,  ..., -11.2254, -13.7492, -13.5460],
         [-13.8155, -13.8155, -13.8155,  ...,  -9.9948, -13.5939, -13.0641],
         [-13.8155, -13.8155, -13.8155,  ..., -10.2373, -13.7537, -13.7698],
         ...,
         [-13.8155, -13.8155, -13.8155,  ..., -13.7461, -13.7135, -13.7663],
         [-13.8155, -13.8155, -13.8155,  ..., -13.7372, -13.6871, -13.7437],
         [-13.8155, -13.8155, -13.8155,  ..., -13.7529, -13.7322, -13.7722]]])

In [13]:
assert torch.log(melspec + 1e-6).shape == logmelbanks.shape

In [14]:
assert torch.allclose(torch.log(melspec + 1e-6), logmelbanks)